In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import model.logistic_regression as lr

In [4]:
df = pd.read_csv("train.csv")
a = df.columns
print(a)
print(df['merchant_category'].value_counts())

with open('DataSolve/output/feature_list.txt', 'r') as f:
    feature_lists = f.read().strip().split('\n')
train_dataset = pd.read_csv("DataSolve/output/train.csv")
X_train = np.asanyarray(train_dataset[feature_lists], dtype=np.float64)
y_train = np.asanyarray(train_dataset['is_fraud'], dtype=np.float64)
val_dataset = pd.read_csv("DataSolve/output/validation.csv")
X_val = np.asanyarray(val_dataset[feature_lists], dtype=np.float64)
y_val = np.asanyarray(val_dataset["is_fraud"], dtype=np.float64)

Index(['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h',
       'distance_from_home_km', 'card_age_months', 'customer_age',
       'account_balance_usd', 'cvv_retry_count', 'velocity_score',
       'time_of_day_hour', 'day_of_week', 'merchant_risk_score',
       'prior_disputes', 'merchant_category', 'card_type', 'auth_method',
       'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant',
       'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch',
       'is_ai_generated_scam_attempt', 'is_fraud'],
      dtype='str')
merchant_category
Online Retail      155097
Groceries          142578
Restaurants        101360
Fuel                59880
Electronics         48693
Streaming           47196
Utilities           42027
Healthcare          35889
Travel              28715
Gaming              23784
Gift Cards           9456
Crypto Exchange      5285
Name: count, dtype: int64


In [3]:
cat = 0
num = 0

with open("describe.txt", "w", encoding="utf-8") as f:
  for i in a:
    f.write("--------" + i + "--------" + "\n")

    if df[i].nunique() < 20:
      f.write(df[i].value_counts().to_string() + "\n")
      cat += 1
    else :
      f.write(df[i].describe().to_string() + "\n")
      num += 1
    f.write("\n")
print(cat)
print(num)

16
10


In [ ]:
log_reg = lr.logistic_regression(n_epochs=2000, reg_lambda=1e-4)
with open('DataSolve/output/feature_list.txt', 'r') as f:
    feature_lists = f.read().strip().split('\n')
train_dataset = pd.read_csv("DataSolve/output/train.csv")
X_train = np.asanyarray(train_dataset[feature_lists], dtype=np.float64)
y_train = np.asanyarray(train_dataset['is_fraud'], dtype=np.float64)
val_dataset = pd.read_csv("DataSolve/output/validation.csv")
X_val = np.asanyarray(val_dataset[feature_lists], dtype=np.float64)
y_val = np.asanyarray(val_dataset["is_fraud"], dtype=np.float64)
log_reg.fit(X_train, y_train, X_val= X_val, y_val=y_val)

In [ ]:
val_dataset = pd.read_csv("DataSolve/output/validation.csv")
X_val = np.asanyarray(val_dataset[feature_lists], dtype=np.float64)
y_val = np.asanyarray(val_dataset["is_fraud"], dtype=np.float64)

# Dự đoán xác suất và nhãn trên tập validation
val_probs = log_reg.predict_proba(X_val)
val_preds = log_reg.predict(X_val, threshold=0.5)

print("Loss train (5 epoch đầu / cuối):", log_reg.history_[:5], log_reg.history_[-5:])
print("Xác suất min / mean / max:", val_probs.min(), val_probs.mean(), val_probs.max())
print("Số ca dự đoán lớp 1 (threshold=0.5):", int(val_preds.sum()), "/ thực tế:", int(y_val.sum()))

In [ ]:
train_hist = np.asarray(log_reg.get_history(), dtype=np.float64)
val_hist = np.asarray(log_reg.get_val_hist(), dtype=np.float64)

print(f"Train epochs: {len(train_hist)} | Val epochs: {len(val_hist)}")

if len(train_hist) > 0:
    print(f"Train loss: đầu={train_hist[0]:.6f} -> cuối={train_hist[-1]:.6f} | min={train_hist.min():.6f}")
    print("Train loss hữu hạn:", np.isfinite(train_hist).all())

if len(val_hist) > 0:
    best_idx = int(np.argmin(val_hist))
    best_epoch = best_idx + 1
    best_val_loss = val_hist[best_idx]

    print(f"Val loss  : đầu={val_hist[0]:.6f} -> cuối={val_hist[-1]:.6f} | min={best_val_loss:.6f} (epoch {best_epoch})")
    print("Val loss hữu hạn  :", np.isfinite(val_hist).all())

    if val_hist[-1] > best_val_loss:
        print(f"Cảnh báo: Val loss tăng sau epoch {best_epoch} ({best_val_loss:.6f} -> {val_hist[-1]:.6f}).")

epochs = np.arange(1, len(train_hist) + 1)
plt.figure(figsize=(8, 4.5))
if len(train_hist) > 0:
    plt.plot(epochs, train_hist, marker="o", markersize=3, linewidth=1.8, label="Train Loss")
if len(val_hist) == len(train_hist) and len(val_hist) > 0:
    plt.plot(epochs, val_hist, marker="s", markersize=3, linewidth=1.8, label="Val Loss")
    best_idx = int(np.argmin(val_hist))
    plt.scatter([best_idx + 1], [val_hist[best_idx]], color="red", zorder=5, label=f"Best Val (Epoch {best_idx + 1})")

plt.title("Train vs Validation Loss (log_reg)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
y_true = y_val.astype(int)

print(f"{'Thresh':>7} | {'Pred_1':>7} | {'TP':>5} | {'Precision':>9} | {'Recall':>8} | {'F1':>8}")
print("-" * 58)

for th in thresholds:
    preds = (val_probs >= th).astype(int)
    tp = int(np.sum((preds == 1) & (y_true == 1)))
    fp = int(np.sum((preds == 1) & (y_true == 0)))
    fn = int(np.sum((preds == 0) & (y_true == 1)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    print(f"{th:7.2f} | {int(preds.sum()):7d} | {tp:5d} | {precision:9.4f} | {recall:8.4f} | {f1:8.4f}")

In [ ]:
log_reg.save("logreg_model_1.npz")

In [ ]:
import numpy as np
import pandas as pd

test_proc = pd.read_csv("DataSolve/output/test_processed.csv")
sub = pd.read_csv("sample_submission.csv")

# Kiểm tra thứ tự id nếu cả hai đều có cột id
if "id" in test_proc.columns and "id" in sub.columns:
    assert (test_proc["id"] == sub["id"]).all(), "Thứ tự id không khớp sample_submission.csv!"

X_test = np.asanyarray(test_proc[feature_lists], dtype=np.float64)
test_probs = log_reg.predict_proba(X_test)

target_col = sub.columns[1]
sub[target_col] = test_probs

# Kiểm tra không có NaN / inf trước khi lưu
assert len(sub) == 299984
assert not sub[target_col].isna().any()
assert np.isfinite(sub[target_col]).all()

sub.to_csv("logistic_result/submission.csv", index=False)
print("Đã xuất submission.csv thành công:", sub.shape)
print("Phân bố xác suất test (min / mean / max):", test_probs.min(), test_probs.mean(), test_probs.max())
print(sub.head())

In [5]:
from model.catboost_wrapper import catboostWrapper
from catboost import Pool

In [ ]:
cb_model = catboostWrapper(iterations=4000, learning_rate=0.03, depth=7, l2_leaf_reg=10)
train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)
# cb_model.fit(X=train_pool, eval_set=val_pool)

In [ ]:
cb_model.fit(X=train_pool, eval_set=val_pool, early_stopping_rounds=2000)

0:	learn: 0.6285802	test: 0.6285259	best: 0.6285259 (0)	total: 5.85ms	remaining: 23.4s
200:	learn: 0.0432220	test: 0.0445271	best: 0.0445271 (200)	total: 1.17s	remaining: 22s
400:	learn: 0.0390847	test: 0.0442394	best: 0.0442142 (338)	total: 2.34s	remaining: 21s
600:	learn: 0.0362269	test: 0.0442952	best: 0.0442083 (435)	total: 3.5s	remaining: 19.8s
800:	learn: 0.0341851	test: 0.0442318	best: 0.0442083 (435)	total: 4.66s	remaining: 18.6s
1000:	learn: 0.0323882	test: 0.0443010	best: 0.0442067 (855)	total: 5.82s	remaining: 17.4s
1200:	learn: 0.0305647	test: 0.0444981	best: 0.0442067 (855)	total: 6.98s	remaining: 16.3s
1400:	learn: 0.0288948	test: 0.0447342	best: 0.0442067 (855)	total: 8.14s	remaining: 15.1s
1600:	learn: 0.0272368	test: 0.0449165	best: 0.0442067 (855)	total: 9.32s	remaining: 14s
1800:	learn: 0.0258219	test: 0.0451168	best: 0.0442067 (855)	total: 10.5s	remaining: 12.8s
2000:	learn: 0.0245369	test: 0.0454843	best: 0.0442067 (855)	total: 11.7s	remaining: 11.7s
2200:	learn: 0

In [12]:
import os
import numpy as np
import pandas as pd
from catboost import Pool

test_proc = pd.read_csv("DataSolve/output/test_processed.csv")
sub = pd.read_csv("sample_submission.csv")

# Kiểm tra thứ tự id nếu cả hai đều có cột id
if "id" in test_proc.columns and "id" in sub.columns:
    assert (test_proc["id"] == sub["id"]).all(), "Thứ tự id không khớp sample_submission.csv!"

# Nếu có cột phân loại (cat_cols), xử lý đồng bộ giống lúc tạo train_pool/val_pool
# Ví dụ:
# cat_cols = [...]
# for col in cat_cols:
#     test_proc[col] = test_proc[col].astype(str).fillna("MISSING")

X_test = test_proc[feature_lists]
test_pool = Pool(X_test)  # Nếu dùng cat_cols thì truyền Pool(X_test, cat_features=cat_cols)

# Dùng cb_model nếu chạy qua wrapper, hoặc model.predict_proba(test_pool)[:, 1] nếu chạy CatBoostClassifier trực tiếp
test_probs = cb_model.predict_proba(test_pool)[:, 1]

target_col = sub.columns[1]
sub[target_col] = test_probs

# Kiểm tra không có NaN / inf trước khi lưu
assert len(sub) == 299984
assert not sub[target_col].isna().any()
assert np.isfinite(sub[target_col]).all()

os.makedirs("catboost_result", exist_ok=True)
out_path = "catboost_result/submission.csv"
sub.to_csv(out_path, index=False)

print("Đã xuất submission.csv thành công:", sub.shape)
print("Phân bố xác suất test (min / mean / max):", test_probs.min(), test_probs.mean(), test_probs.max())
print(sub.head())

Đã xuất submission.csv thành công: (299984, 2)
Phân bố xác suất test (min / mean / max): 1.4330092484920037e-05 0.01574967726985437 0.9938467763509528
       id  is_fraud
0  699960  0.000270
1  699961  0.000099
2  699962  0.081387
3  699963  0.000689
4  699964  0.004338
